
# Task 7 – Understanding User Preferences and Preparing Interaction Data

## Objective

The objective of this task is to understand how user preferences and interactions can be represented as data in a recommendation system.

In this task, we create a sample user interaction dataset containing multiple users and TED Talks. We also convert this dataset into a User-Item Interaction Matrix and perform basic data analysis.

Since actual user interaction data is unavailable, we use simulated data for this learning exercise.


In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

In [17]:
talks = pd.read_csv(r"C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\processed\ted_talks_cleaned.csv")

print("Dataset Shape:", talks.shape)

display(talks.head())

Dataset Shape: (2550, 9)


,title,description,tags,main_speaker,speaker_occupation,event,views,url,transcript
0,do schools kill creativity?,sir ken robinson makes an entertaining and pro...,children creativity culture dance education pa...,Ken Robinson,Author/educator,TED2006,47227110,https://www.ted.com/talks/ken_robinson_says_sc...,"good morning. how are you? it's been great, ha..."
1,averting the climate crisis,with the same humor and humanity he exuded in ...,alternative energy cars climate change culture...,Al Gore,Climate advocate,TED2006,3200520,https://www.ted.com/talks/al_gore_on_averting_...,"thank you so much, chris. and it's truly a gre..."
2,simplicity sells,new york times columnist david pogue takes aim...,computers entertainment interface design media...,David Pogue,Technology columnist,TED2006,1636292,https://www.ted.com/talks/david_pogue_says_sim...,"hello voice mail, my old friend. i've called f..."
3,greening the ghetto,"in an emotionally charged talk, macarthur-winn...",macarthur grant activism business cities envir...,Majora Carter,Activist for environmental justice,TED2006,1697550,https://www.ted.com/talks/majora_carter_s_tale...,if you're here today — and i'm very happy that...
4,the best stats you've ever seen,you've never seen data presented like this. wi...,africa asia google demo economics global devel...,Hans Rosling,Global health expert; data visionary,TED2006,12005869,https://www.ted.com/talks/hans_rosling_shows_t...,"about 10 years ago, i took on the task to teac..."


In [18]:
# Reset index to create consistent Talk IDs
talks = talks.reset_index(drop=True)

# Select the first 50 TED Talks
talks = talks.head(50).copy()

# Generate unique Talk IDs
talks["talk_id"] = [f"T{i+1:03d}" for i in range(len(talks))]

# Create a small talk catalog
talk_catalog = talks[["talk_id", "title", "main_speaker"]]

print("Total TED Talks:", len(talk_catalog))

talk_catalog.head()

Total TED Talks: 50


,talk_id,title,main_speaker
0,T001,do schools kill creativity?,Ken Robinson
1,T002,averting the climate crisis,Al Gore
2,T003,simplicity sells,David Pogue
3,T004,greening the ghetto,Majora Carter
4,T005,the best stats you've ever seen,Hans Rosling


In [19]:
# Set random seed to get reproducible results
np.random.seed(42)

# Create 30 simulated users
user_ids = [f"U{i+1:03d}" for i in range(30)]

# Store all interactions
interaction_data = []

for user_id in user_ids:

    # Randomly select 15 to 25 talks for each user
    number_of_talks = np.random.randint(15, 26)

    selected_talks = np.random.choice(
        talk_catalog["talk_id"],
        size=number_of_talks,
        replace=False
    )

    for talk_id in selected_talks:

        # Generate a rating between 1 and 5
        rating = np.random.randint(1, 6)

        # Determine whether the user liked or disliked the talk
        liked = rating >= 4
        disliked = rating <= 2

        # Randomly decide whether the user saved the talk
        saved = np.random.choice([True, False], p=[0.3, 0.7])

        interaction_data.append({
            "user_id": user_id,
            "talk_id": talk_id,
            "rating": rating,
            "liked": liked,
            "disliked": disliked,
            "watched": True,
            "saved": saved
        })

# Convert the interaction data into a DataFrame
interactions = pd.DataFrame(interaction_data)

# Display the first five interactions
interactions.head()

,user_id,talk_id,rating,liked,disliked,watched,saved
0,U001,T044,5,True,False,True,True
1,U001,T041,4,True,False,True,False
2,U001,T047,4,True,False,True,False
3,U001,T013,3,False,False,True,False
4,U001,T025,4,True,False,True,True


In [20]:
print("Total interactions:", len(interactions))
print("Number of unique users:", interactions["user_id"].nunique())
print("Number of unique talks:", interactions["talk_id"].nunique())

print("\nDataset shape:", interactions.shape)

print("\nMissing values:")
print(interactions.isnull().sum())

print("\nDuplicate user-talk interactions:")
print(interactions.duplicated(subset=["user_id", "talk_id"]).sum())

Total interactions: 597
Number of unique users: 30
Number of unique talks: 50

Dataset shape: (597, 7)

Missing values:
user_id     0
talk_id     0
rating      0
liked       0
disliked    0
watched     0
saved       0
dtype: int64

Duplicate user-talk interactions:
0


In [21]:
# Create the interaction folder if it does not exist
Path("../data/interaction").mkdir(parents=True, exist_ok=True)

# Save the interaction dataset
interactions.to_csv(
    "../data/interaction/user_interactions.csv",
    index=False
)

print("Interaction dataset saved successfully!")

Interaction dataset saved successfully!


In [22]:
# Create the User-Item Interaction Matrix
user_item_matrix = interactions.pivot(
    index="user_id",
    columns="talk_id",
    values="rating"
)

# Display the matrix
user_item_matrix

talk_id,T001,T002,T003,T004,T005,T006,T007,T008,T009,T010,...,T041,T042,T043,T044,T045,T046,T047,T048,T049,T050
user_id,,,,,,,,,,,,,,,,,,,,,
U001,NaN,NaN,NaN,2.0,1.0,NaN,2.0,NaN,5.0,NaN,...,4.0,NaN,NaN,5.0,NaN,NaN,4.0,4.0,3.0,2.0
U002,NaN,4.0,NaN,1.0,NaN,3.0,NaN,NaN,NaN,5.0,...,NaN,NaN,5.0,5.0,1.0,4.0,1.0,2.0,3.0,NaN
U003,3.0,5.0,NaN,NaN,NaN,NaN,NaN,5.0,NaN,4.0,...,4.0,3.0,NaN,NaN,3.0,3.0,NaN,1.0,NaN,NaN
U004,NaN,4.0,NaN,NaN,NaN,NaN,5.0,NaN,4.0,NaN,...,4.0,NaN,2.0,4.0,5.0,4.0,NaN,NaN,NaN,NaN
U005,NaN,4.0,NaN,2.0,NaN,5.0,5.0,4.0,NaN,1.0,...,NaN,NaN,3.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN
U006,1.0,2.0,NaN,NaN,NaN,2.0,NaN,5.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,5.0,4.0,5.0,NaN
U007,NaN,4.0,NaN,1.0,5.0,NaN,3.0,NaN,2.0,NaN,...,NaN,1.0,5.0,NaN,1.0,NaN,4.0,NaN,NaN,NaN
U008,NaN,4.0,NaN,1.0,NaN,NaN,5.0,2.0,5.0,2.0,...,1.0,NaN,2.0,3.0,NaN,NaN,NaN,NaN,3.0,NaN
U009,NaN,4.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN,2.0,...,3.0,4.0,NaN,NaN,5.0,4.0,1.0,NaN,2.0,3.0


In [23]:
print("Number of users:", user_item_matrix.shape[0])
print("Number of talks:", user_item_matrix.shape[1])

print("\nUser-Item Matrix:")
display(user_item_matrix.head())

print("\nMissing values in the matrix:")
print(user_item_matrix.isnull().sum().sum())

Number of users: 30
Number of talks: 50

User-Item Matrix:


talk_id,T001,T002,T003,T004,T005,T006,T007,T008,T009,T010,...,T041,T042,T043,T044,T045,T046,T047,T048,T049,T050
user_id,,,,,,,,,,,,,,,,,,,,,
U001,NaN,NaN,NaN,2.0,1.0,NaN,2.0,NaN,5.0,NaN,...,4.0,NaN,NaN,5.0,NaN,NaN,4.0,4.0,3.0,2.0
U002,NaN,4.0,NaN,1.0,NaN,3.0,NaN,NaN,NaN,5.0,...,NaN,NaN,5.0,5.0,1.0,4.0,1.0,2.0,3.0,NaN
U003,3.0,5.0,NaN,NaN,NaN,NaN,NaN,5.0,NaN,4.0,...,4.0,3.0,NaN,NaN,3.0,3.0,NaN,1.0,NaN,NaN
U004,NaN,4.0,NaN,NaN,NaN,NaN,5.0,NaN,4.0,NaN,...,4.0,NaN,2.0,4.0,5.0,4.0,NaN,NaN,NaN,NaN
U005,NaN,4.0,NaN,2.0,NaN,5.0,5.0,4.0,NaN,1.0,...,NaN,NaN,3.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN



Missing values in the matrix:
903


In [24]:
# Count how many talks each user has rated
user_rating_counts = interactions.groupby("user_id")["rating"].count()

# Display users with the most ratings
print("Users with the most ratings:")
display(user_rating_counts.sort_values(ascending=False).head(10))

Users with the most ratings:


user_id
U002    25
U009    25
U025    24
U021    24
U011    23
U012    23
U016    23
U020    23
U029    23
U008    22
Name: rating, dtype: int64

In [25]:
# Calculate average rating and number of ratings for each talk
talk_rating_analysis = interactions.groupby("talk_id").agg(
    average_rating=("rating", "mean"),
    total_ratings=("rating", "count")
)

# Sort talks by average rating
highest_rated_talks = talk_rating_analysis.sort_values(
    by=["average_rating", "total_ratings"],
    ascending=[False, False]
)

# Add TED Talk titles
highest_rated_talks = highest_rated_talks.merge(
    talk_catalog[["talk_id", "title"]],
    on="talk_id"
)

print("Highest-rated TED Talks:")
display(highest_rated_talks.head(10))

Highest-rated TED Talks:


,talk_id,average_rating,total_ratings,title
0,T046,3.700000,10,how technology evolves
1,T016,3.600000,15,an 11-year-old's magical violin
2,T028,3.571429,7,the freakonomics of crack dealing
3,T021,3.545455,11,the birth of the open-source learning revolution
4,T002,3.538462,13,averting the climate crisis
5,T007,3.538462,13,letting go of god
6,T029,3.500000,12,"choice, happiness and spaghetti sauce"
7,T048,3.500000,12,8 secrets of success
8,T026,3.444444,9,chemical scum that dream of distant quasars
9,T011,3.416667,12,my wish: a call for open-source architecture


In [26]:
# Count interactions for each TED Talk
talk_interaction_counts = interactions.groupby("talk_id").size()

# Sort by interaction count
most_interacted_talks = talk_interaction_counts.sort_values(
    ascending=False
).reset_index(name="total_interactions")

# Add TED Talk titles
most_interacted_talks = most_interacted_talks.merge(
    talk_catalog[["talk_id", "title"]],
    on="talk_id"
)

print("Most-interacted TED Talks:")
display(most_interacted_talks.head(10))

Most-interacted TED Talks:


,talk_id,total_interactions,title
0,T009,17,let's teach religion -- all religion -- in sch...
1,T043,16,how juries are fooled by statistics
2,T044,16,why people believe weird things
3,T047,15,fight injustice with raw video
4,T016,15,an 11-year-old's magical violin
5,T014,15,the radical promise of the multi-touch interface
6,T032,14,meet the future of cancer research
7,T018,14,simple designs to save a life
8,T010,14,a life of purpose
9,T008,14,behind the design of seattle's library


In [27]:
print("Total likes:", interactions["liked"].sum())
print("Total dislikes:", interactions["disliked"].sum())
print("Total watched talks:", interactions["watched"].sum())
print("Total saved talks:", interactions["saved"].sum())

Total likes: 243
Total dislikes: 246
Total watched talks: 597
Total saved talks: 185


In [28]:
# Check whether the dataset contains the expected number of users and talks
assert interactions["user_id"].nunique() == 30
assert interactions["talk_id"].nunique() == 50

# Check that ratings are between 1 and 5
assert interactions["rating"].between(1, 5).all()

# Check that each user-talk pair is unique
assert not interactions.duplicated(
    subset=["user_id", "talk_id"]
).any()

print("All dataset checks passed successfully!")

All dataset checks passed successfully!



## Understanding User Interaction Data

User interaction data represents how users interact with items in a recommendation system.

In our project, users interact with TED Talks in different ways.

### Important Concepts

1. **User ID:** A unique identifier assigned to each user.

2. **TED Talk ID:** A unique identifier assigned to each TED Talk.

3. **Ratings:** Numerical values given by users to express their interest in a TED Talk. For example, ratings from 1 to 5.

4. **Likes and Dislikes:** Indicate whether a user likes or dislikes a particular TED Talk.

5. **Watch History:** Records the TED Talks watched by a user.

6. **Saved or Bookmarked Talks:** TED Talks saved by users to watch later.

7. **User Preferences:** Represent the interests and choices of individual users.

8. **User-Item Interaction Matrix:** A table representing interactions between users and items. Rows represent users, columns represent TED Talks, and values represent ratings.


# Observations

1. A simulated interaction dataset was created for 30 users and 50 TED Talks.
2. Each user interacted with a different selection of TED Talks.
3. Users provided ratings between 1 and 5.
4. The dataset also contains information about likes, dislikes, watched talks, and saved talks.
5. The User-Item Interaction Matrix represents users as rows and TED Talks as columns.
6. Missing values indicate that a user has not rated a particular TED Talk.
7. The interaction dataset can be used as input for collaborative filtering.

## Conclusion

The user interaction dataset and User-Item Interaction Matrix were successfully created. This dataset will be used in the next notebook to implement a collaborative filtering recommendation system.